# Запуск на суперкомьютере

In [8]:
import os
import sys
import time
from collections import Counter

sys.path.insert(0, os.path.abspath("/opt/clients"))
import osiris  # noqa
from IPython.display import display, clear_output, Markdown  # noqa


def logs_training_jobs(time_sleep=5):
    runs = [x["job"] for x in osiris.list()["jobs"]]
    while True:
        for run in runs:
            if osiris.state(run)["state"] == "Running":
                try:
                    display(
                        Markdown(f"### **{run.split('pytorchjob')[-1][1:]}**"),
                        Markdown(osiris.logs(
                            run, tail_lines=1
                        )["pods"][0]["logs"][1].replace("'", "")),
                        Markdown("---")
                    )
                except:  # noqa
                    pass
                print()
        time.sleep(time_sleep)
        clear_output(wait=True)


def delete_job(job_name):
    osiris.delete(job_name)


def delete_all_inactive_jobs():
    for job in osiris.list()["jobs"]:
        if job["state"] in ["Failed", "Succeeded"]:
            delete_job(job["job"])


def jobs_stat():
    print(Counter(job["state"] for job in osiris.list()["jobs"]))

In [13]:
# delete_all_inactive_jobs()
jobs_stat()

Counter({'Succeeded': 3})


# Sequential Inference

## Inference sequences month part by month part 8m states

In [12]:
sequences_dir = "/home/datalab/projects/avatar/avatards/rusakov/8m_chains_2026_03_31"  # noqa
dir_to_save = "/home/datalab/projects/avatar/avatards/rusakov/8m_states_2026_03_31"  # noqa

config_dir = "/home/datalab/nfs/avatar_campaign/experiments/sbercampaign_pilot/configs/prom/sequence"  # noqa
config_name = "inference_sequence"

prefixes = os.listdir(sequences_dir + "/" + os.listdir(sequences_dir)[0])
month_parts = [x for x in os.listdir(sequences_dir) if x.startswith("month_part")]  # noqa
partitions = [(month_part, prefix) for month_part in month_parts for prefix in prefixes]  # noqa

print(month_parts)
print(partitions)

['month_part=2026-03-31']
[('month_part=2026-03-31', 'part_1'), ('month_part=2026-03-31', 'part_2'), ('month_part=2026-03-31', 'part_3'), ('month_part=2026-03-31', 'part_4'), ('month_part=2026-03-31', 'part_5'), ('month_part=2026-03-31', 'part_6'), ('month_part=2026-03-31', 'part_7'), ('month_part=2026-03-31', 'part_8'), ('month_part=2026-03-31', 'part_9'), ('month_part=2026-03-31', 'part_10'), ('month_part=2026-03-31', 'part_11'), ('month_part=2026-03-31', 'part_12'), ('month_part=2026-03-31', 'part_13'), ('month_part=2026-03-31', 'part_14'), ('month_part=2026-03-31', 'part_15'), ('month_part=2026-03-31', 'part_16'), ('month_part=2026-03-31', 'part_17'), ('month_part=2026-03-31', 'part_18')]


In [13]:
for month_part, prefix in partitions:
    job_name = month_part.split("=")[-1] + "-" + prefix
    print(job_name)

    job = osiris.create(
        name=job_name.replace("_", "-"),
        image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",  # noqa
        restart=False,
        command=[
            "accelerate", "launch", "-m", "avatar.inference"
        ],
        args=[
            f"--config-dir={config_dir}",
            f"--config-name={config_name}",
            f"metrics.test_metrics.path_to_save='{dir_to_save}/{month_part}'",  # noqa
            f"metrics.test_metrics.prefix={prefix}",
            f"test_dataloader.dataset.path='{sequences_dir}/{month_part}/{prefix}'"  # noqa
        ],
        envs={
            "PYTHONPATH": "/home/datalab/nfs/pckgs",
            "OMP_NUM_THREADS": "7",
            "NCCL_DEBUG": "INFO",
        },
        num_nodes=1,
        num_gpus=1,
        type="pytorchjob",
    )

    display(job)
    time.sleep(1)

2026-03-31-part_1


{'job': 'training-21937299-pytorchjob-2026-03-31-part-1',
 'uuid': 'e8291bfe-a3cc-4f12-b8f7-f50f24e3da33',
 'state': 'Created',
 'time': '2026-04-09T12:01:22Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

2026-03-31-part_2


{'job': 'training-21937299-pytorchjob-2026-03-31-part-2',
 'uuid': '374e8f6b-8163-4557-960d-016af55244e2',
 'state': 'Created',
 'time': '2026-04-09T12:01:23Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

2026-03-31-part_3


...

{'job': 'training-21937299-pytorchjob-2026-03-31-part-16',
 'uuid': 'bf2e81f3-2776-4d12-adb7-b4e3d38cd6a6',
 'state': 'Created',
 'time': '2026-04-09T12:01:40Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

2026-03-31-part_17


{'job': 'training-21937299-pytorchjob-2026-03-31-part-17',
 'uuid': '36a94fb8-2032-450a-809d-3c5451e27568',
 'state': 'Created',
 'time': '2026-04-09T12:01:41Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

2026-03-31-part_18


{'job': 'training-21937299-pytorchjob-2026-03-31-part-18',
 'uuid': '82236f5e-1425-4f9d-81b3-c7e5fca20ede',
 'state': 'Created',
 'time': '2026-04-09T12:01:43Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

In [9]:
osiris.logs("training-21937299-pytorchjob-2026-03-31-part-1", tail_lines=10)

{'job': 'training-21937299-pytorchjob-2026-03-31-part-1',
 'pods': [{'pod': 'training-21937299-pytorchjob-2026-03-31-part-1-master-0',
   'state': 'Failed',
   'logs': ['2026-04-09T11:53:35.570106986Z             type: categorical',
    '2026-04-09T11:53:35.570106986Z             event_id: 1',
    '2026-04-09T11:53:35.570106986Z           txn_evt_attr_2:',
    '2026-04-09T11:53:35.570106986Z             n_classes: 4',
    '2026-04-09T11:53:35.570106986Z             type: categorical',
    '2026-04-09T11:53:35.570106986Z             event_id: 1',
    '2026-04-09T11:53:35.570106986Z           txn_evt_attr_3:',
    '2026-04-09T11:53:35.570106986Z             n_classes: 44',
    '2026-04-09T11:53:35.570106986Z             type: categorical',
    '2026-04-09T11:53:35.570106986Z             event_id: 1',
    '2026-04-09T11:53:35.570106986Z     backbone:',
    '2026-04-09T11:53:35.570106986Z       _target_: transformers.GPT2Model',
    '2026-04-09T11:53:35.570106986Z       config:',
    '2026

## Inference downstream chains. (partitioned by product_name, month_part)

In [10]:
sequences_dir = "/home/datalab/projects/avatards/avatar/rusakov/january_pilot/pilot/calibration_chains_2025_11"
dir_to_save = "/home/datalab/projects/avatards/avatar/rusakov/january_pilot/pilot/42m_states_2025_11"

config_dir = "/home/datalab/nfs/avatar_main/avatar_fm/experiments/nikita/configs/large"  # noqa
config_name = "large_inference"

products = os.listdir(sequences_dir)
all_paths = [(product, os.listdir(sequences_dir + "/" + product)) for product in products]  # noqa

# all_paths = [("all", [x]) for x in products]
all_paths

[('product_name=bo_response', ['month_part=2025-09-30']),
 ('product_name=cc_response', ['month_part=2025-09-30']),
 ('product_name=mg_response', ['month_part=2025-09-30']),
 ('product_name=pds_response', ['month_part=2025-09-30']),
 ('product_name=sa_response', ['month_part=2025-09-30']),
 ('product_name=tdbase_response', ['month_part=2025-09-30'])]

In [11]:
for product, month_parts in all_paths:
    for month_part in month_parts:
        data_path = f"{sequences_dir}/{product}/{month_part}" # inferennce downstream chains product by product
        # data_path = f"{sequences_dir}/{month_part}"  # inference pilot slice 42m
        path_to_save = f"{dir_to_save}/{product}/{month_part}"

        job_name = (
            product.split("=")[-1].split("_")[0] + "-" +
            month_part.split("=")[-1]
        )
        print(job_name)
        job = osiris.create(
            name=job_name,
            image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",  # noqa
            restart=False,
            command=[
                "accelerate", "launch", "-m", "avatar.inference"
            ],
            args=[
                f"--config-dir={config_dir}",
                f"--config-name={config_name}",
                f"metrics.test_metrics.path_to_save='{path_to_save}'",  # noqa
                # f"metrics.test_metrics.prefix={prefix}",
                f"test_dataloader.dataset.path='{data_path}'"  # noqa
            ],
            envs={
                "PYTHONPATH": "/home/datalab/nfs/pckgs",
                "OMP_NUM_THREADS": "7",
                "NCCL_DEBUG": "INFO",
            },
            num_nodes=1,
            num_gpus=1,
            type="pytorchjob",
        )
        display(job)
        time.sleep(1)

bo-2025-09-30


{'job': 'training-21937299-pytorchjob-bo-2025-09-30',
 'uuid': '2313a696-e30a-4143-a530-fe624c881059',
 'state': 'Created',
 'time': '2025-12-11T19:00:41Z',
 'projects': [{'project_name': 'CDS Avatar DS',
   'project_key': 'avatards',
   'project_claim_name': 'project-avatards',
   'project_mount_path': '/home/datalab/projects/avatards'}]}

cc-2025-09-30


{'job': 'training-21937299-pytorchjob-cc-2025-09-30',
 'uuid': '4d6682e2-fc3d-4b47-afb5-fa2189b07cad',
 'state': 'Created',
 'time': '2025-12-11T19:00:42Z',
 'projects': [{'project_name': 'CDS Avatar DS',
   'project_key': 'avatards',
   'project_claim_name': 'project-avatards',
   'project_mount_path': '/home/datalab/projects/avatards'}]}

mg-2025-09-30


...

{'job': 'training-21937299-pytorchjob-pds-2025-09-30',
 'uuid': '18799acd-3e84-48c1-ba18-2cf90cb224dc',
 'state': 'Created',
 'time': '2025-12-11T19:00:44Z',
 'projects': [{'project_name': 'CDS Avatar DS',
   'project_key': 'avatards',
   'project_claim_name': 'project-avatards',
   'project_mount_path': '/home/datalab/projects/avatards'}]}

sa-2025-09-30


{'job': 'training-21937299-pytorchjob-sa-2025-09-30',
 'uuid': 'da49d977-b4c2-44dd-8e8e-44c932105650',
 'state': 'Created',
 'time': '2025-12-11T19:00:45Z',
 'projects': [{'project_name': 'CDS Avatar DS',
   'project_key': 'avatards',
   'project_claim_name': 'project-avatards',
   'project_mount_path': '/home/datalab/projects/avatards'}]}

tdbase-2025-09-30


{'job': 'training-21937299-pytorchjob-tdbase-2025-09-30',
 'uuid': '67e35504-833f-41e6-ac06-21d3e0975fd6',
 'state': 'Created',
 'time': '2025-12-11T19:00:46Z',
 'projects': [{'project_name': 'CDS Avatar DS',
   'project_key': 'avatards',
   'project_claim_name': 'project-avatards',
   'project_mount_path': '/home/datalab/projects/avatards'}]}

# Inference TabNN

## Pilot Inference по партициям (inference.py)

In [ ]:
data_dirs = [
    x
    for x in os.listdir("/home/datalab/projects/avatar/avatards/rusakov/april_pilot/2026_04_01/prom/inference_tabular_2026_04_01/splited_tabular/split_type=test")
    if not x.startswith(".ipynb")
]

script = "inference"
postfix_job_name = "prom-inf"
# config_names = ["bo", "td", "sa", "pds"]
config_names = ["bo", "sa"]
config_dir = "/home/datalab/nfs/avatar_campaign/experiments/sbercampaign_pilot/configs/prom/tabular/inference/"  # NOQA

for data_dir in data_dirs:
    for config_name in config_names:
        job_name = config_name.replace("_", "-")

        job = osiris.create(
            name=job_name + f"-{postfix_job_name}-inf",
            image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",  # NOQA
            restart=False,
            command=[
                "accelerate", "launch", "--num_processes=1", "-m", f"avatar.{script}"
            ],
            args=[
                f"--config-dir={config_dir}",
                f"--config-name={config_name}",
                f"test_dataloader.dataset.path='/home/datalab/projects/avatar/avatards/rusakov/april_pilot/2026_04_01/prom/inference_tabular_2026_04_01/splited_tabular/split_type=test/{data_dir}'",
                f"metrics.test_metrics.prefix"
            ],
            envs={
                "PYTHONPATH": "/home/datalab/nfs/pckgs",
                "OMP_NUM_THREADS": "7",
                "NCCL_DEBUG": "INFO",
            },
            # pool="inference",
            num_nodes=1,
            num_gpus=1,
            type="pytorchjob",
        )

    display(job)
    time.sleep(1)

['partition_2',
 'partition_3',
 'partition_4',
 'partition_5',
 'partition_6',
 'partition_7',
 'partition_8',
 'partition_1']

## Pilot Inference(inference.py)

In [11]:
script = "inference"
postfix_job_name = "prom-inf"
# config_names = ["bo", "td", "sa", "pds"]
config_names = ["bo", "sa", "td"]
config_dir = "/home/datalab/nfs/avatar_campaign/experiments/sbercampaign_pilot/configs/prom/tabular/inference"  # NOQA

for config_name in config_names:
    job_name = config_name.replace("_", "-")

    job = osiris.create(
        name=job_name + f"-{postfix_job_name}-inf",
        image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",  # NOQA
        restart=False,
        command=[
            "accelerate", "launch", "--num_processes=1", "-m", f"avatar.{script}"
        ],
        args=[
            f"--config-dir={config_dir}",
            f"--config-name={config_name}"
        ],
        envs={
            "PYTHONPATH": "/home/datalab/nfs/pckgs",
            "OMP_NUM_THREADS": "7",
            "NCCL_DEBUG": "INFO",
        },
        # pool="inference",
        num_nodes=1,
        num_gpus=1,
        type="pytorchjob",
    )

    display(job)
    time.sleep(1)

{'job': 'training-21937299-pytorchjob-bo-prom-inf-inf',
 'uuid': 'b84836fb-9367-46c6-aa95-327c96def4bc',
 'state': 'Created',
 'time': '2026-04-16T10:28:25Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

{'job': 'training-21937299-pytorchjob-sa-prom-inf-inf',
 'uuid': '85fb19ed-7392-455f-97f6-fdb55326e840',
 'state': 'Created',
 'time': '2026-04-16T10:28:26Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

{'job': 'training-21937299-pytorchjob-td-prom-inf-inf',
 'uuid': '2bbf7be1-4a8e-4d5f-afbd-8add5a075b0d',
 'state': 'Created',
 'time': '2026-04-16T10:28:27Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

In [48]:
osiris.logs("training-21937299-pytorchjob-sa-prom-inf-v2inf", tail_lines=10)

{'job': 'training-21937299-pytorchjob-sa-prom-inf-v2inf',
 'pods': [{'pod': 'training-21937299-pytorchjob-sa-prom-inf-v2inf-master-0',
   'state': 'Succeeded',
   'logs': ['2026-04-05T13:14:21.242807934Z W0405 13:14:21.242000 139734378739520 torch/_dynamo/convert_frame.py:762] [2/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.',
    '2026-04-05T13:14:21.244188689Z W0405 13:14:21.244000 139734378739520 torch/_dynamo/convert_frame.py:762] [3/8] torch._dynamo hit config.cache_size_limit (8)',
    "2026-04-05T13:14:21.244188689Z W0405 13:14:21.244000 139734378739520 torch/_dynamo/convert_frame.py:762] [3/8]    function: 'apply_expanded_mask' (/home/datalab/nfs/pckgs/avatar/nn/utils/agg.py:71)",
    "2026-04-05T13:14:21.244188689Z W0405 13:14:21.244000 139734378739520 torch/_dynamo/convert_frame.py:762] [3/8]    last reason: tensor 'L['states']' size mismatch at index 0. expected 1625, actual 1706",
    '2026-04-05T13:14:21.2441886

## Multi-GPU training TabNN

In [4]:
script = "train"
postfix_job_name = "prom-training"
config_names = ["bo", "td", "sa"]
config_dir = "/home/datalab/nfs/avatar_campaign/experiments/sbercampaign_pilot/configs/prom/tabular/train"  # NOQA

for config_name in config_names:
    job_name = config_name.replace("_", "-")

    job = osiris.create(
        name=job_name + f"-{postfix_job_name}",
        image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",  # NOQA
        restart=False,
        command=[
            "accelerate", "launch", "-m", f"avatar.{script}"
        ],
        args=[
            f"--config-dir={config_dir}",
            f"--config-name={config_name}",
        ],
        envs={
            "PYTHONPATH": "/home/datalab/nfs/pckgs",
            "OMP_NUM_THREADS": "7",
            "NCCL_DEBUG": "INFO",
        },
        num_nodes=1,
        num_gpus=4,
        pool="public",
        type="pytorchjob",
    )

    display(job)
    time.sleep(1)

{'job': 'training-21937299-pytorchjob-bo-prom-training',
 'uuid': '1f50965c-bd80-4141-b841-1f8d45b6db21',
 'state': 'Created',
 'time': '2026-04-13T11:27:31Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

{'job': 'training-21937299-pytorchjob-td-prom-training',
 'uuid': '9ac821ae-d820-4143-a228-e790b116628c',
 'state': 'Created',
 'time': '2026-04-13T11:27:32Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

{'job': 'training-21937299-pytorchjob-sa-prom-training',
 'uuid': '7e0db70b-d509-4c75-a91f-db201b00617b',
 'state': 'Created',
 'time': '2026-04-13T11:27:34Z',
 'projects': [{'project_name': 'CDS Avatar',
   'project_key': 'avatar',
   'project_claim_name': 'project-avatar',
   'project_mount_path': '/home/datalab/projects/avatar'}]}

In [55]:
osiris.logs("training-21937299-pytorchjob-td-prom-training", tail_lines=20)

{'job': 'training-21937299-pytorchjob-td-prom-training',
 'pods': [{'pod': 'training-21937299-pytorchjob-td-prom-training-master-0',
   'state': 'Running',
   'logs': ['2026-04-10T17:31:27.388432879+03:00 ',
    '2026-04-10T17:31:27.388497428+03:00 ',
    'Validation step:  80%|████████  | 112/140 [00:24<00:05,  4.81it/s]\x1b[A2026-04-10T17:31:27.589700195+03:00 ',
    '2026-04-10T17:31:27.589767235+03:00 ',
    'Validation step:  81%|████████  | 113/140 [00:25<00:05,  4.86it/s]\x1b[A2026-04-10T17:31:27.807289342+03:00 ',
    '2026-04-10T17:31:27.807353982+03:00 ',
    'Validation step:  81%|████████▏ | 114/140 [00:25<00:05,  4.78it/s]\x1b[A2026-04-10T17:31:28.018663574+03:00 ',
    '2026-04-10T17:31:28.018729304+03:00 ',
    'Validation step:  82%|████████▏ | 115/140 [00:25<00:05,  4.76it/s]\x1b[A2026-04-10T17:31:28.230552203+03:00 ',
    '2026-04-10T17:31:28.230620793+03:00 ',
    'Validation step:  83%|████████▎ | 116/140 [00:25<00:05,  4.75it/s]\x1b[A2026-04-10T17:31:28.444782414+0